In [22]:
import pandas as pd

df = pd.read_csv("../data/used_cars_data.csv", nrows=200_000)  # sample first — full file is 3M rows
df.shape

C:\Users\mrsha\AppData\Local\Temp\ipykernel_16240\774318924.py:3: DtypeWarning: Columns (0: dealer_zip) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/used_cars_data.csv", nrows=200_000)  # sample first — full file is 3M rows


(200000, 66)

In [23]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 66 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   vin                      200000 non-null  str    
 1   back_legroom             190542 non-null  str    
 2   bed                      1205 non-null    str    
 3   bed_height               20252 non-null   str    
 4   bed_length               20252 non-null   str    
 5   body_type                199210 non-null  str    
 6   cabin                    3372 non-null    str    
 7   city                     200000 non-null  str    
 8   city_fuel_economy        168272 non-null  float64
 9   combine_fuel_economy     0 non-null       float64
 10  daysonmarket             200000 non-null  int64  
 11  dealer_zip               200000 non-null  object 
 12  description              194926 non-null  str    
 13  engine_cylinders         194164 non-null  str    
 14  engine_displace

In [24]:
df[['price', 'mileage', 'year', 'make_name', 'model_name', 'trim_name',
    'horsepower', 'body_type', 'has_accidents', 'is_new']].describe(include='all')

,price,mileage,year,make_name,model_name,trim_name,horsepower,body_type,has_accidents,is_new
count,2.000000e+05,191990.000000,200000.000000,200000,200000,193022,189844.000000,199210,111477,200000
unique,NaN,NaN,NaN,63,977,4827,NaN,9,2,2
top,NaN,NaN,NaN,Ford,F-150,Limited 4WD,NaN,SUV / Crossover,False,False
freq,NaN,NaN,NaN,28990,6019,4220,NaN,103803,91093,109339
mean,3.036053e+04,31124.117876,2017.552075,NaN,NaN,NaN,244.913487,NaN,NaN,NaN
std,2.274858e+04,43103.621000,3.888968,NaN,NaN,NaN,87.053326,NaN,NaN,NaN
min,2.990000e+02,0.000000,1930.000000,NaN,NaN,NaN,70.000000,NaN,NaN,NaN
25%,1.829500e+04,7.000000,2017.000000,NaN,NaN,NaN,175.000000,NaN,NaN,NaN
50%,2.699900e+04,14021.000000,2019.000000,NaN,NaN,NaN,241.000000,NaN,NaN,NaN
75%,3.818900e+04,43776.000000,2020.000000,NaN,NaN,NaN,295.000000,NaN,NaN,NaN


In [25]:
df.isnull().sum().sort_values(ascending=False)

is_certified               200000
combine_fuel_economy       200000
vehicle_damage_category    200000
bed                        198795
cabin                      196628
is_oemcpo                  188045
is_cpo                     182756
bed_length                 179748
bed_height                 179748
owner_count                 93520
salvage                     88523
theft_title                 88523
frame_damaged               88523
fleet                       88523
has_accidents               88523
isCab                       88523
franchise_make              45234
highway_fuel_economy        31728
city_fuel_economy           31728
torque                      30427
main_picture_url            28212
power                       28212
interior_color              22021
major_options               12372
engine_displacement         10156
horsepower                  10156
width                        9458
back_legroom                 9458
front_legroom                9458
height        

In [26]:
pd.set_option('display.max_rows', None)
df.isnull().sum().sort_values(ascending=False)


is_certified               200000
combine_fuel_economy       200000
vehicle_damage_category    200000
bed                        198795
cabin                      196628
is_oemcpo                  188045
is_cpo                     182756
bed_length                 179748
bed_height                 179748
owner_count                 93520
salvage                     88523
theft_title                 88523
frame_damaged               88523
fleet                       88523
has_accidents               88523
isCab                       88523
franchise_make              45234
highway_fuel_economy        31728
city_fuel_economy           31728
torque                      30427
main_picture_url            28212
power                       28212
interior_color              22021
major_options               12372
engine_displacement         10156
horsepower                  10156
width                        9458
back_legroom                 9458
front_legroom                9458
height        

In [27]:
drop_cols = [
    'is_certified', 'combine_fuel_economy', 'vehicle_damage_category',
    'bed', 'cabin', 'bed_length', 'bed_height', 'is_oemcpo', 'isCab',
    'main_picture_url', 'listing_id', 'sp_id', 'sp_name', 'dealer_zip',
    'latitude', 'longitude', 'vin'
]
df = df.drop(columns=drop_cols)
df.shape

(200000, 49)

In [28]:
df = df[df['is_new'] == False].copy()
df.shape

(109339, 49)

In [29]:
history_cols = ['salvage', 'theft_title', 'frame_damaged', 'fleet', 'has_accidents']

for col in history_cols:
    df[col] = df[col].fillna(False)

df['owner_count'] = df['owner_count'].fillna(df['owner_count'].median())

In [30]:
numeric_impute_cols = [
    'horsepower', 'torque', 'power', 'highway_fuel_economy',
    'city_fuel_economy', 'engine_displacement'
]

for col in numeric_impute_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # convert strings to NaN
    df[col] = df[col].fillna(df[col].median())


In [31]:
required_cols = ['price', 'mileage', 'year', 'make_name', 'model_name']
df = df.dropna(subset=required_cols)
df.shape

(108195, 49)

In [32]:
df = df[
    (df['price'].between(500, 150_000)) &
    (df['year'] >= 1990) &
    (df['mileage'] <= 300_000)
]
df.shape

(107697, 49)

In [33]:
df[['price', 'mileage', 'year', 'horsepower']].describe()

,price,mileage,year,horsepower
count,107697.000000,107697.000000,107697.000000,107697.000000
mean,22299.224231,55188.677707,2015.528557,244.000864
std,13316.262953,44216.995940,3.763751,82.252830
min,599.000000,50.000000,1990.000000,70.000000
25%,13895.000000,24515.000000,2014.000000,176.000000
50%,19799.000000,39337.000000,2017.000000,241.000000
75%,28695.000000,77978.000000,2018.000000,295.000000
max,150000.000000,300000.000000,2021.000000,808.000000


In [34]:
import re

def scrub_description(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\[!@@.*?@@!\]", "", text)
    text = text.replace("\n", " ").strip()
    return text[:400]   # shortened from 1000

def build_full(row):
    parts = [
        f"{int(row['year'])} {row['make_name']} {row['model_name']} {row['trim_name'] or ''}".strip(),
        f"Body type: {row['body_type']}",
        f"Mileage: {int(row['mileage']):,} miles",
        f"Horsepower: {int(row['horsepower'])} hp",
        f"Fuel type: {row['fuel_type']}" if pd.notna(row['fuel_type']) else "",
        f"Transmission: {row['transmission_display']}" if pd.notna(row['transmission_display']) else "",
        f"Accident history: {'Yes' if row['has_accidents'] else 'No reported accidents'}",
        f"Frame damage: {'Yes' if row['frame_damaged'] else 'No'}",
        scrub_description(row['description']),
    ]
    return "\n".join(p for p in parts if p)

df['full'] = df.apply(build_full, axis=1)
print(df['full'].iloc[0])

2020 Land Rover Range Rover Evoque P300 R-Dynamic SE AWD
Body type: SUV / Crossover
Mileage: 254 miles
Horsepower: 296 hp
Fuel type: Gasoline
Transmission: 9-Speed Automatic Overdrive
Accident history: No reported accidents
Frame damage: No
4-Wheel Disc Brakes,A/C,ABS,Adjustable Steering Wheel,All Wheel Drive,Aluminum Wheels,Auto-Dimming Rearview Mirror,Automatic Headlights,Automatic Parking,Auxiliary Audio Input,Back-Up Camera,Bluetooth Connection,Brake Actuated Limited Slip Differential,Brake Assist,Bucket Seats,Cargo Shade,Child Safety Locks,Climate Control,Cross-Traffic Alert,Cruise Control,Daytime Running Lights,Driver Adjustabl


In [35]:
import sys
sys.path.append("..")  # so we can import car_pricer from notebooks/

from auto_pricer.car import Car

def na(val):
    return None if pd.isna(val) else val

def row_to_car(row):
    return Car(
        title=f"{int(row['year'])} {row['make_name']} {row['model_name']}",
        make=row['make_name'],
        model=row['model_name'],
        trim=na(row.get('trim')),
        year=int(row['year']),
        mileage=float(row['mileage']),
        price=float(row['price']),
        horsepower=na(row.get('horsepower')),
        body_type=na(row.get('body_type')),
        fuel_type=na(row.get('fuel_type')),
        transmission=na(row.get('transmission')),
        has_accidents=na(row.get('has_accidents')),
        frame_damaged=na(row.get('frame_damaged')),
        full=na(row.get('full')),
    )


cars = [row_to_car(row) for _, row in df.iterrows()]
len(cars)

107697

In [36]:
for i, car in enumerate(cars):
    car.id = i

cars[0]

<2020 Land Rover Range Rover Evoque = $84399.0>

In [37]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(cars, test_size=0.1, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

len(train), len(val), len(test)

(96927, 5385, 5385)

In [38]:
print(train[0].full)
print(train[0])

2017 Jeep Compass Limited 4WD
Body type: SUV / Crossover
Mileage: 17,933 miles
Horsepower: 180 hp
Fuel type: Gasoline
Transmission: Automatic
Accident history: No reported accidents
Frame damage: No
CARFAX 1-Owner, LOW MILES - 17,933! EPA 30 MPG Hwy/22 MPG City! Moonroof, Heated Leather Seats, Nav System, WHEELS: 19 X 7.5 POLISHED/BLACK POC... WHEELS: 19 X 7.5 POLISHED/BLACK POCKETS ALUMINUM, Bluetooth, Back-Up Camera, 4x4, NAVIGATION GROUP, ADVANCED SAFETY & LIGHTING GROUP KEY FEATURES INCLUDELeather Seats, 4x4, Heated Driver Seat, Back-Up Camera, Bluetooth. OPTION PACKAGESPOWER FRONT/FIXED 
title='2017 Jeep Compass' make='Jeep' model='Compass' trim=None year=2017 mileage=17933.0 price=24497.0 horsepower=180.0 body_type='SUV / Crossover' fuel_type='Gasoline' transmission='A' has_accidents=False frame_damaged=False full='2017 Jeep Compass Limited 4WD\nBody type: SUV / Crossover\nMileage: 17,933 miles\nHorsepower: 180 hp\nFuel type: Gasoline\nTransmission: Automatic\nAccident history: N

In [39]:
from dotenv import load_dotenv
from huggingface_hub import login
import os

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [40]:
from pydantic import BaseModel

from datasets import Dataset, DatasetDict, load_dataset
from typing import Self

class Car(BaseModel):
    # ... (keep everything you already have) ...

    @staticmethod
    def push_to_hub(dataset_name: str, train: list["Car"], val: list["Car"], test: list["Car"]):
        DatasetDict({
            "train": Dataset.from_list([c.model_dump() for c in train]),
            "validation": Dataset.from_list([c.model_dump() for c in val]),
            "test": Dataset.from_list([c.model_dump() for c in test]),
        }).push_to_hub(dataset_name)

    @classmethod
    def from_hub(cls, dataset_name: str) -> tuple[list[Self], list[Self], list[Self]]:
        ds = load_dataset(dataset_name)
        return (
            [cls.model_validate(row) for row in ds["train"]],
            [cls.model_validate(row) for row in ds["validation"]],
            [cls.model_validate(row) for row in ds["test"]],
        )

In [41]:
username = "ShahidHKhan"  # replace with your actual HF username
dataset_name = f"{username}/cars_raw"

Car.push_to_hub(dataset_name, train, val, test)
print(f"Pushed to https://huggingface.co/datasets/{dataset_name}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  4.22ba/s]
Processing Files (1 / 1): 100%|██████████| 28.5MB / 28.5MB, 1.05MB/s  
New Data Upload: 100%|██████████| 28.5MB / 28.5MB, 1.05MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:21<00:00, 21.73s/ shards]
Setting num_proc from 1 back to 1 for the validation split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 49.55ba/s]
Processing Files (1 / 1): 100%|██████████| 1.62MB / 1.62MB,  142kB/s  
New Data Upload: 100%|██████████| 1.62MB / 1.62MB,  142kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.96s/ shards]
Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:0

Pushed to https://huggingface.co/datasets/ShahidHKhan/cars_raw


In [42]:
loaded_train, loaded_val, loaded_test = Car.from_hub(dataset_name)
print(len(loaded_train), len(loaded_val), len(loaded_test))
print(loaded_train[0])

Generating test split: 100%|██████████| 5385/5385 [00:00<00:00, 386533.76 examples/s]


96927 5385 5385

